In [ ]:
#无参数寻优最终版
import os
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE
from sklearn.base import BaseEstimator, ClassifierMixin, clone

import xgboost as xgb
from scipy.signal import savgol_filter
from scipy.stats import pearsonr


# ===================== 核心配置 =====================
train_path = r"H:\图像标记\上下采样\三类精准平衡后_训练集.xlsx"
test_path = r"H:\图像标记\610-0.52SAVI.xlsx"
save_dir = r"H:\图像标记\传统特征选择结果"
os.makedirs(save_dir, exist_ok=True)

CV_FOLDS = 5
SEED = 42
n_bands = 178
MAX_FEATURES_TO_TEST = 178   # 允许全部波段逐个引入


# ===================== 1. 加载数据 =====================
def load_data():
    train_df = pd.read_excel(train_path, engine='openpyxl')
    test_df = pd.read_excel(test_path, engine='openpyxl')

    X_train = train_df.iloc[:, 1:].values
    y_train = train_df.iloc[:, 0].values
    X_test = test_df.iloc[:, 1:].values
    y_test = test_df.iloc[:, 0].values

    X_train = X_train[:, :n_bands]
    X_test = X_test[:, :n_bands]

    return X_train, y_train, X_test, y_test


# ===================== 2. 预处理 =====================
def preprocess_minmax(X_train, X_test):
    scaler = MinMaxScaler((0, 1))
    return scaler.fit_transform(X_train), scaler.transform(X_test)

def preprocess_snv(X_train, X_test):
    def snv(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)
    return np.apply_along_axis(snv, 1, X_train), np.apply_along_axis(snv, 1, X_test)

def preprocess_sg(X_train, X_test):
    return savgol_filter(X_train, 11, 2, axis=1), savgol_filter(X_test, 11, 2, axis=1)

def preprocess_wd(X_train, X_test):
    def denoise(x):
        return np.convolve(x, np.ones(3)/3, mode='same')
    return np.apply_along_axis(denoise, 1, X_train), np.apply_along_axis(denoise, 1, X_test)

preprocessors = {
    "minmax": preprocess_minmax,
    "snv": preprocess_snv,
    "sg": preprocess_sg,
    "wd": preprocess_wd
}


# ===================== 3. 特征选择 =====================
def fs_uev(X_train):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    cv = std / (mean + 1e-8)
    return np.argsort(cv)[::-1]

def fs_cars(X_train, y_train):
    pls = PLSRegression(n_components=5)
    pls.fit(X_train, pd.get_dummies(y_train).values)
    coef = np.sum(np.abs(pls.coef_), axis=1)
    return np.argsort(coef)[::-1]

def fs_spa(X_train, y_train):
    corr = [abs(pearsonr(X_train[:, i], y_train)[0]) for i in range(n_bands)]
    return np.argsort(corr)[::-1]

def fs_mrmr(X_train, y_train):
    corr = [abs(pearsonr(X_train[:, i], y_train)[0]) for i in range(n_bands)]
    return np.argsort(corr)[::-1]

def fs_svm_rfe(X_train, y_train):
    svc = SVC(kernel='linear', random_state=SEED)
    rfe = RFE(estimator=svc, n_features_to_select=1, step=5)
    rfe.fit(X_train, y_train)
    return np.argsort(rfe.ranking_)

def fs_xgb(X_train, y_train):
    model = xgb.XGBClassifier(
        n_estimators=50,
        random_state=SEED,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
    model.fit(X_train, y_train)
    return np.argsort(model.feature_importances_)[::-1]

feature_selectors = {
    "UEV": (fs_uev, False),
    "CARS": (fs_cars, True),
    "SPA": (fs_spa, True),
    "mRMR": (fs_mrmr, True),
    "SVM-RFE": (fs_svm_rfe, True),
    "XGBoost": (fs_xgb, True)
}


# ===================== 4. 修正版 PLS-DA =====================
class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=10):
        self.n_components = n_components
        self.pls = None
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        max_comp = min(X.shape[1], X.shape[0] - 1)
        n_comp = min(self.n_components, max_comp)
        self.pls = PLSRegression(n_components=n_comp)
        self.pls.fit(X, pd.get_dummies(y).values)
        return self

    def predict(self, X):
        pred = self.pls.predict(X)
        return self.classes_[np.argmax(pred, axis=1)]

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


classifiers = {
    "SVC": SVC(random_state=SEED),
    "KNN": KNeighborsClassifier(),
    "RF": RandomForestClassifier(n_estimators=50, random_state=SEED),
    "PLS-DA": PLSDA(),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=50,
        random_state=SEED,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
}


# ===================== 5. 逐个引入特征 =====================
def select_features(X_train, y_train, sorted_idx, clf):
    selected = []
    best_score = 0
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    for idx in sorted_idx[:MAX_FEATURES_TO_TEST]:
        trial = selected + [idx]
        X_trial = X_train[:, trial]

        clf_temp = clone(clf)

        if hasattr(clf_temp, "n_components"):
            max_comp = min(X_trial.shape[1], X_trial.shape[0] - 1)
            clf_temp.n_components = min(clf_temp.n_components, max_comp)

        try:
            score = cross_val_score(clf_temp, X_trial, y_train, cv=skf).mean()
        except:
            score = 0

        if score > best_score:
            selected.append(idx)
            best_score = score

    return selected, best_score


# ===================== 6. 模型评估 =====================
def evaluate_model(X_train, y_train, X_test, y_test, selected, clf):
    X_train_sel = X_train[:, selected]
    X_test_sel = X_test[:, selected]

    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    clf_temp = clone(clf)
    if hasattr(clf_temp, "n_components"):
        max_comp = min(X_train_sel.shape[1], X_train_sel.shape[0] - 1)
        clf_temp.n_components = min(clf_temp.n_components, max_comp)

    cv_acc = cross_val_score(clf_temp, X_train_sel, y_train, cv=skf).mean()

    clf_temp.fit(X_train_sel, y_train)
    y_pred = clf_temp.predict(X_test_sel)

    class_acc = []
    for cls in [0,1,2]:
        mask = y_test == cls
        if np.sum(mask) > 0:
            class_acc.append(accuracy_score(y_test[mask], y_pred[mask]))
        else:
            class_acc.append(0)

    return round(cv_acc,4), round(np.mean(class_acc),4), class_acc


# ===================== 7. 主流程 =====================
def main():
    X_train, y_train, X_test, y_test = load_data()
    all_results = []

    for pre_name, pre_fun in preprocessors.items():
        print(f"\n===== 预处理：{pre_name} =====")
        X_train_pre, X_test_pre = pre_fun(X_train.copy(), X_test.copy())

        for fs_name, (fs_fun, need_y) in feature_selectors.items():
            print(f"\n--- 特征选择：{fs_name} ---")

            sorted_idx = fs_fun(X_train_pre, y_train) if need_y else fs_fun(X_train_pre)

            for clf_name, clf in classifiers.items():
                print(f"正在运行：{pre_name}-{fs_name}-{clf_name}")

                selected, _ = select_features(X_train_pre, y_train, sorted_idx, clf)

                if len(selected) == 0:
                    continue

                cv_acc, test_acc, class_acc = evaluate_model(
                    X_train_pre, y_train, X_test_pre, y_test, selected, clf
                )

                result = {
                    "预处理方法": pre_name,
                    "特征选择方法": fs_name,
                    "分类器": clf_name,
                    "筛选特征数量": len(selected),
                    "筛选特征列表": ",".join([f"Band_{i+1}" for i in selected]),
                    "五折交叉验证精度": cv_acc,
                    "测试集平均精度（宏平均）": test_acc,
                    "水稻精度（0类）": round(class_acc[0],4),
                    "稗草精度（1类）": round(class_acc[1],4),
                    "千金子精度（2类）": round(class_acc[2],4)
                }

                all_results.append(result)
                print(f"✅ 特征数：{len(selected)} | CV:{cv_acc} | Test:{test_acc}")

    result_df = pd.DataFrame(all_results)
    save_path = os.path.join(save_dir, "传统+逐个波段结果plsda.xlsx")
    result_df.to_excel(save_path, index=False)

    print("\n🎉 全部完成！")
    print(f"保存路径：{save_path}")
    print(result_df.head())


if __name__ == "__main__":
    main()